# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library. All dataset entities (record sets, fields, columns) are referenced using their `@id` as per the Croissant specification.

### Dataset Source
The dataset source (Croissant schema JSON-LD):
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Install mlcroissant if necessary
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and initialize the `mlcroissant` Dataset object.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)
print(f"Successfully loaded the dataset Croissant schema.")

# Show basic metadata
md = dataset.metadata
print("\nDataset Name:", md.name)
print("Version:", getattr(md, 'version', 'unknown'))
print("Identifier:", getattr(md, 'identifier', 'unknown'))
print("Description:\n", md.description)
print("\nLicense:", getattr(md, 'license', None), "\n")

## 2. Data Overview
Review all record set `@id`s, field `@id`s, and additional metadata using the Croissant schema. All references use the `@id` for consistent referencing and loading.

In [ ]:
print("Available record sets and their fields (by @id):\n")

record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets defined in schema. Attempting to list fields of main dataset (flat file).\n")
    # In many datasets, a single record set may be implicit named as 'records' or 'data'. Attempt to infer.
    # Try to get candidate record set @id(s)
    # Try to print the field IDs available (if possible):
    try:
        fields = dataset.metadata.fields
        for field in fields:
            print(f"Field @id: {field['@id']}")
    except Exception as e:
        print("Could not list fields:", e)
    # Attempt to show one record (if fields detection failed):
    try:
        sample = next(dataset.records())
        print("\nExample record keys:", list(sample.keys()))
    except Exception as e:
        print("Could not load a sample record:", e)
else:
    for recset in record_sets:
        print(f"RecordSet @id: {recset['@id']}")
        for field in recset.get('fields', []):
            print(f"  Field @id: {field['@id']}")

## 3. Data Extraction
Load data from the default/main record set (or the flat main table if there's only one).

We must specify the `record_set` argument if there are multiple, but for single main datasets, this is optional. Field identifiers are always `@id` strings.

In [ ]:
# Try identifying the main record set. For datasets with a single main table, this is often implicit.
# mlcroissant lets you use record_set=None for the main data in that case.

# Load all records (may take time if large)
df = pd.DataFrame(dataset.records())
print(f"Loaded {len(df)} records. Columns (@id):\n{df.columns.tolist()}")

# Show the first 5 rows
df.head()

## 4. Exploratory Data Analysis (EDA)
We demonstrate how to filter, normalize, and group by fields using only the Croissant `@id` of each column.

First, we inspect the columns to pick relevant fields. _You may need to adjust the `numeric_field_id` and `group_field_id` below based on output above._

In [ ]:
# Example: Use field @id values. Replace as appropriate for your dataset.
print("All available fields (columns) by @id:\n", df.columns.tolist())

# Sample numeric field: attempt to select first numeric-looking field
import re
numeric_candidate = None
for col in df.columns:
    # Heuristics for fields commonly in mass spec tables
    if any(s in col.lower() for s in ["abundance", "peptide", "count", "sum", "coverage", "mw", "mass", "intensity"]):
        numeric_candidate = col
        break
if not numeric_candidate:
    # fallback: pick first column with numeric values
    for col in df.select_dtypes(include=[np.number]).columns:
        numeric_candidate = col
        break
if not numeric_candidate:
    numeric_candidate = df.columns[0]  # fallback

print(f"\nChosen numeric field (by @id): {numeric_candidate}")

# If present, also try to find a grouping/categorical field
group_candidate = None
for col in df.columns:
    if any(s in col.lower() for s in ["description", "accession", "sample", "protein", "group"]):
        group_candidate = col
        break
if not group_candidate:
    # Fallback: pick first non-numeric column
    for col in df.select_dtypes(include=[object]).columns:
        group_candidate = col
        break
if not group_candidate:
    group_candidate = df.columns[0]

print(f"Chosen group field (by @id): {group_candidate}\n")

# Ensure numeric; coerce errors
df[numeric_candidate] = pd.to_numeric(df[numeric_candidate], errors='coerce')

threshold = df[numeric_candidate].quantile(0.9)  # top 10% as threshold
filtered_df = df[df[numeric_candidate] > threshold]

print(f"Filtered records with {numeric_candidate} > {threshold:.2f} (top 10%): {len(filtered_df)} records\n")

# Normalize the field, add _normalized column
filtered_df[f"{numeric_candidate}_normalized"] = (
    filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()
) / filtered_df[numeric_candidate].std()

print(f"Normalized '{numeric_candidate}' for filtered records:")
display_cols = [numeric_candidate, f"{numeric_candidate}_normalized"]
print(filtered_df[display_cols].head())

# Group by the group_candidate if there are multiple values
if group_candidate in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_candidate)[numeric_candidate].mean().reset_index()
    print(f"\nGroup mean of '{numeric_candidate}' by '{group_candidate}' (top 5):")
    print(grouped_df.head())

## 5. Visualization
Visualize distribution and relationships using field `@id` names. Example: plot histogram of a numeric field, scatter plot if multiple numeric fields present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_candidate].dropna(), kde=True, bins=30)
plt.title(f"Histogram of {numeric_candidate}")
plt.xlabel(numeric_candidate)
plt.show()

# If there are at least 2 numeric columns, plot a scatter
numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric_fields) > 1:
    plt.figure(figsize=(7, 5))
    sns.scatterplot(x=numeric_fields[0], y=numeric_fields[1], data=df, alpha=0.7)
    plt.title(f"Scatter: {numeric_fields[0]} vs {numeric_fields[1]}")
    plt.xlabel(numeric_fields[0])
    plt.ylabel(numeric_fields[1])
    plt.show()

## 6. Conclusion
We demonstrated loading mass spectrometry data defined by a Croissant schema, and explored, summarized, and visualized it using only field and record set `@id` references. `mlcroissant` enables standardized FAIR data processing workflows that can be adapted to any schema-compliant biomedical or scientific dataset.

To extend this notebook:
- Experiment with other field or record set `@id` values as displayed above.
- Integrate with downstream ML, stats, or bioinformatics analyses, always referencing fields by their `@id`.